In [35]:
import numpy as np
import pandas as pd
import yaml
import os
from matplotlib import pyplot as plt
from hsa_hopper.kinematics import KinematicParameters
from hsa_hopper.controller import HopController
from hsa_hopper.hsa_model import HSAModel
kin_params = KinematicParameters(.07,.15,.3,-.005)
root = r'C:\Users\Joseph Sullivan\Documents\hsa_hopper_control\data\aim2\temp_storage'
dataframes = {}
hardware_configs = {}
dynamics_params = {}
controller_params = {}
for category in os.listdir(root):
    dataframes[category] = {}
    hardware_configs[category] = {}
    dynamics_params[category] = {}
    controller_params[category] = {}
    category_folder = os.path.join(root,category)
    total_mass = int(category.split('_')[-1][:-1])/1000
    if 'no_hsa' in category:
        continue
    for trial in os.listdir(category_folder):
        trial_folder = os.path.join(category_folder, trial)
        # load config files
        with open(os.path.join(trial_folder,'hardware_config.yaml')) as f:
            hardware_configs[category][trial] = yaml.load(f,yaml.Loader)
        with open(os.path.join(trial_folder,'experiment_config.yaml')) as f:
            experiment_config = yaml.load(f,yaml.Loader)
            dynamics_params[category][trial] = experiment_config['dynamics_params']
            controller_params[category][trial] = experiment_config['controller']
        # load (smoothed) data files
        trial_df = pd.read_csv(os.path.join(trial_folder,'data.csv'))
        m_cart = experiment_config['dynamics_params']['m_cart']
        m_foot = experiment_config['dynamics_params']['m_foot']
        trial_df['total_mass'] = m_cart+m_foot 
        trial_df['psi'] = experiment_config['dynamics_params']['psi']
        # drop flight modes from the dataframes
        trial_df = trial_df[trial_df['mode'] == HopController._STANCE]
        dataframes[category][trial] = trial_df

actuator_model_path = r'C:\Users\Joseph Sullivan\Documents\hsa_hopper_control\notebooks\actuator_model.yaml'
with open(actuator_model_path, 'r') as f:
    actuator_model = yaml.load(f, yaml.Loader)

char_model_path = r'C:\Users\Joseph Sullivan\Documents\hsa_hopper_control\notebooks\hsa_model.yaml'
with open(char_model_path, 'r') as f:
    hsa_char_model = HSAModel.make_from_dict(yaml.load(f, yaml.Loader))

In [36]:
from hsa_hopper.collocation import PiecewiseInterpolation
from hsa_hopper.controller import HopController
stance_refs = {}
torque_refs = {}
for category, trials in controller_params.items():
    for trial, params in trials.items():
        x_interp_dict= params['x_interp'][HopController._STANCE]
        u_interp_dict = params['u_interp'][HopController._STANCE]
        stance_refs[trial] = x_interp =  PiecewiseInterpolation.make_from_dict(x_interp_dict)
        torque_refs[trial] = u_interp = PiecewiseInterpolation.make_from_dict(u_interp_dict)
        df = dataframes[category][trial]
        df['x_ref'] = np.zeros(len(df))
        df['xdot_ref'] = np.zeros(len(df))
        df['xddot_ref'] = np.zeros(len(df))
        df['torque_ref'] = np.zeros(len(df))
        for idx in df['hop_idx'].unique():
            frame = df['hop_idx'] == idx
            t = df.loc[frame,'t_s'].to_numpy()
            t = t-t[0]
            df.loc[frame, 't_s'] = t
            df.loc[frame,'x_ref'] = np.array([x_interp.evaluate(_t) for _t in t])
            df.loc[frame,'xdot_ref'] = np.array([x_interp.evaluate(_t, ord=1) for _t in t])
            df.loc[frame,'xddot_ref'] = np.array([x_interp.evaluate(_t, ord=2) for _t in t])
            df.loc[frame,'torque_ref'] = np.array([u_interp.evaluate(_t) for _t in t])

In [37]:
for category, trials in dataframes.items():
    for trial, df in trials.items():
        x = df['x_rad']
        x_ref = df['x_ref']
        x_errors = x-x_ref
        print(f'mean error in x: {x_errors.mean()}')
        print(f'stddev error in x: {x_errors.std()}')
        print(f'sum of squared errors x: {(x_errors**2).sum()}')
        print(f'R^2 of x_ref predicting x: {1-(x_errors**2).sum()/(x**2).sum()}')
        torque = df['torque']
        torque_ref = df['torque_ref']
        torque_errors = torque-torque_ref
        print(f'mean error in u: {torque_errors.mean()}')
        print(f'stddev error in u: {torque_errors.std()}')
        print(f'sum of squared errors u: {(torque_errors**2).sum()}')
        print(f'R^2 of torque_ref predicting torque: {1-(torque_errors**2).sum()/(torque**2).sum()}')

mean error in x: 0.012251429013661323
stddev error in x: 0.02374319625255664
sum of squared errors x: 2.988985118947391
R^2 of x_ref predicting x: 0.9886515869234338
mean error in u: 0.43231958415821303
stddev error in u: 0.38386333049116844
sum of squared errors u: 1399.6970068299036
R^2 of torque_ref predicting torque: 0.8308795637567121
mean error in x: 0.009718983955821209
stddev error in x: 0.025681643039171267
sum of squared errors x: 3.1608852561072145
R^2 of x_ref predicting x: 0.991237157442078
mean error in u: 0.46810802200553847
stddev error in u: 0.36497535270170756
sum of squared errors u: 1477.1954071453297
R^2 of torque_ref predicting torque: 0.8296739661541938
mean error in x: 0.007728347280008775
stddev error in x: 0.0244515177098127
sum of squared errors x: 2.7573935928352777
R^2 of x_ref predicting x: 0.993335081756363
mean error in u: 0.503377113693922
stddev error in u: 0.3459574796318906
sum of squared errors u: 1564.557267321572
R^2 of torque_ref predicting torqu

In [39]:
all_data = pd.concat([df for condition, trials in dataframes.items() for trial, df in trials.items()])

In [ ]:
x = all_data['x_rad']*180/np.pi
x_ref = all_data['x_ref']
x_errors = x-x_ref
torque = all_data['torque']
torque_ref = all_data['torque_ref']
torque_errors = torque-torque_ref

print(f'mean error in x: {x_errors.mean()}')
print(f'stddev error in x: {x_errors.std()}')
print(f'sum of squared errors x: {(x_errors**2).sum()}')
print(f'R^2 of x_ref predicting x: {1-(x_errors**2).sum()/(x**2).sum()}')
print(f'mean error in u: {torque_errors.mean()}')
print(f'stddev error in u: {torque_errors.std()}')
print(f'sum of squared errors u: {(torque_errors**2).sum()}')
print(f'R^2 of u_ref predicting u: {1-(torque_errors**2).sum()/(torque**2).sum()}')

mean error in x: 3.000398782917917
stddev error in x: 3.1212331331649303
sum of squared errors x: 1546391.8664830057
R^2 of x_ref predicting x: 0.911128049213451
mean error in u: 0.6055668419486256
stddev error in u: 0.45253620805835293
sum of squared errors u: 47147.991833656546
R^2 of u_ref predicting u: 0.8479457549105318


In [45]:
# now examine this by condition (mass)
masses = np.array(all_data['total_mass'].unique())
xerr_mean = np.zeros_like(masses)
xerr_std = np.zeros_like(masses)
xerr_rsqr = np.zeros_like(masses)
uerr_mean = np.zeros_like(masses)
uerr_std = np.zeros_like(masses)
uerr_rsqr = np.zeros_like(masses)

for i, mass in enumerate(masses):
    frame = all_data['total_mass'] == mass
    x = all_data.loc[frame,'x_rad']*180/np.pi
    x_ref = all_data.loc[frame,'x_ref']*180/np.pi
    x_errors = x-x_ref
    xerr_mean[i] = x_errors.mean()
    xerr_std[i] = x_errors.std()
    xerr_rsqr[i] = 1-(x_errors**2).sum() / (x**2).sum()
    torque = all_data.loc[frame,'torque']
    torque_ref = all_data.loc[frame,'torque_ref']
    torque_errors = torque-torque_ref
    uerr_mean[i] = torque_errors.mean()
    uerr_std[i] = torque_errors.std()
    uerr_rsqr[i] = 1-(torque_errors**2).sum() / (torque**2).sum()
print(f'mass: {masses}')
print(f'x error mean: {xerr_mean}')
print(f'x error std: {xerr_std}')
print(f'x error R^2: {xerr_rsqr}')
print(f'u error mean: {uerr_mean}')
print(f'u error std: {uerr_std}')
print(f'u error R^2: {uerr_rsqr}')

mass: [1.3 1.5 1.7 1.9 2.1]
x error mean: [0.63432171 1.63115827 2.95187647 4.23947567 5.56666822]
x error std: [1.42430788 1.8287228  2.39570847 2.9427455  3.64847509]
x error R^2: [0.99029452 0.97380084 0.93230919 0.86192946 0.73660166]
u error mean: [0.47397746 0.55735236 0.61208595 0.66457412 0.72109101]
u error std: [0.37849287 0.37802045 0.42703837 0.49736142 0.52157097]
u error R^2: [0.82269938 0.83500277 0.84667242 0.85192243 0.86099967]
